### Ejemplo: Cómo usar los modelos guardados con otro dataset

A continuación se muestra cómo cargar cualquiera de los modelos entrenados y usarlos para predecir sobre un nuevo dataset sin necesidad de reentrenar.

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def cargar_y_usar_modelo(nombre_modelo, nuevo_dataset_path):
    """
    Función para cargar un modelo específico y usarlo con un nuevo dataset
    
    Args:
        nombre_modelo: 'random_forest', 'logistic_regression', o 'svm_sigmoid'
        nuevo_dataset_path: ruta al nuevo dataset CSV
    """
    
    print(f"CARGANDO MODELO: {nombre_modelo.upper()}")
    print("=" * 50)
    
    # Cargar el pipeline de preprocesamiento
    vectorizador = joblib.load("modelos_entrenados/vectorizador_tfidf.joblib")
    
    # Cargar SVD si se usó
    try:
        svd_transformer = joblib.load("modelos_entrenados/svd_transformer.joblib")
        usar_svd = True
        print("✓ SVD transformer cargado")
    except FileNotFoundError:
        svd_transformer = None
        usar_svd = False
        print("✓ No se usó SVD en el entrenamiento")
    
    # Cargar el modelo específico
    model_path = f"modelos_entrenados/{nombre_modelo}_optimizado.joblib"
    modelo = joblib.load(model_path)
    print(f"✓ Modelo {nombre_modelo} cargado desde: {model_path}")
    
    # Cargar metadata para ver hiperparámetros
    metadata = joblib.load("modelos_entrenados/experiment_metadata.joblib")
    model_info = None
    for key, value in metadata["model_results"].items():
        if key.lower().replace(" ", "_").replace("(", "").replace(")", "") == nombre_modelo:
            model_info = value
            break
    
    if model_info:
        print(f"✓ Hiperparámetros: {model_info['best_params']}")
        print(f"✓ F1-Score original: {model_info['best_score']:.4f}")
    
    # Cargar nuevo dataset
    print(f"\nCargando nuevo dataset: {nuevo_dataset_path}")
    try:
        nuevo_data = pd.read_csv(nuevo_dataset_path)
        print(f"✓ Dataset cargado: {nuevo_data.shape[0]} filas")
    except Exception as e:
        print(f"❌ Error cargando dataset: {e}")
        return None
    
    # Verificar que tenga las columnas necesarias
    if 'Sentence' not in nuevo_data.columns:
        print("❌ El dataset debe tener una columna 'Sentence'")
        return None
    
    # Preprocesar nuevo dataset
    print("\nPreprocesando nuevo dataset...")
    X_nuevo = vectorizador.transform(nuevo_data["Sentence"].astype(str))
    print(f"✓ Vectorización TF-IDF completada: {X_nuevo.shape}")
    
    if usar_svd:
        X_nuevo = svd_transformer.transform(X_nuevo)
        print(f"✓ Transformación SVD completada: {X_nuevo.shape}")
    
    # Realizar predicciones
    print("\nRealizando predicciones...")
    predicciones = modelo.predict(X_nuevo)
    probabilidades = modelo.predict_proba(X_nuevo)[:, 1] if hasattr(modelo, 'predict_proba') else None
    
    # Mostrar resumen de predicciones
    pred_counts = np.bincount(predicciones)
    print("✓ Predicciones completadas:")
    print(f"  - Benignas (0): {pred_counts[0]} ({pred_counts[0]/len(predicciones)*100:.1f}%)")
    if len(pred_counts) > 1:
        print(f"  - SQLi (1): {pred_counts[1]} ({pred_counts[1]/len(predicciones)*100:.1f}%)")
    else:
        print("  - SQLi (1): 0 (0.0%)")
    
    # Si el nuevo dataset tiene etiquetas, evaluar rendimiento
    if 'Label' in nuevo_data.columns:
        print(f"\n{'='*50}")
        print("EVALUACIÓN EN NUEVO DATASET (con etiquetas reales):")
        print("="*50)
        
        y_real = nuevo_data['Label'].astype(int)
        print(classification_report(y_real, predicciones, 
                                  target_names=['Benigna (0)', 'SQLi (1)']))
        
        # Matriz de confusión
        cm = confusion_matrix(y_real, predicciones)
        print("\nMatriz de confusión:")
        print(f"  TN: {cm[0,0]}, FP: {cm[0,1]}")
        print(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")
    
    return {
        'modelo': modelo,
        'predicciones': predicciones,
        'probabilidades': probabilidades,
        'dataset': nuevo_data
    }

In [ ]:
# EJEMPLO DE USO:
# Descomenta las líneas siguientes para probar con tu dataset

# # Usar Random Forest
# resultado_rf = cargar_y_usar_modelo('random_forest', './data/nuevo_dataset.csv')

# # Usar Logistic Regression
# resultado_lr = cargar_y_usar_modelo('logistic_regression', './data/nuevo_dataset.csv')

# # Usar SVM
# resultado_svm = cargar_y_usar_modelo('svm_sigmoid', './data/nuevo_dataset.csv')

print("✅ FUNCIÓN cargar_y_usar_modelo() LISTA PARA USAR")
print("Descomenta las líneas de ejemplo arriba para probar con tu dataset")
